# Topic modeling on persona_text (real seal package)
Unzip `seal_pkg.zip` (contains `seal/preprocessing`, `seal/prompting`, `seal/topic_modeling`) into this notebook's working directory, then run top to bottom.

In [4]:
# --- Unpack the seal package (upload seal_pkg.zip to this Colab session first) ---
!unzip -o -q seal_pkg.zip

unzip:  cannot find or open seal_pkg.zip, seal_pkg.zip.zip or seal_pkg.zip.ZIP.


In [2]:
# --- Dependencies ---
!pip install tomotopy plotly rbo gensim kneed python-dotenv ollama openai scikit-learn tqdm --quiet
!python -m spacy download en_core_web_lg --quiet

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')


In [1]:
import logging, os, json
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import sparse

from seal.topic_modeling import LDATopicModel
from seal.preprocessing import SimpleTMPreprocessor

from openai import OpenAI

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)s  %(message)s")
logger = logging.getLogger("persona_tm")

## Load data and build `text` from `persona_text`

In [11]:
df = pd.read_csv("base_data_prompt.csv")
df = df[df["prompt_type"] == "original_post"]

df["text"] = df["prompt_text"].fillna("").astype(str)
df = df[df["text"] != ""].reset_index(drop=True)

if "id" not in df.columns:
    df["id"] = df.index.astype(str)

print(len(df), "documents")
df.head()

1000 documents


,Unnamed: 0,prompt_type,prompt_id,prompt_text,text,id
0,0,original_post,c7g6mt,Dinnertime dilemma\n\nWe had BLTs for dinner. ...,Dinnertime dilemma\n\nWe had BLTs for dinner. ...,0
1,2,original_post,7dy89e,Would it be wrong if I sold this ring?\n\nArou...,Would it be wrong if I sold this ring?\n\nArou...,1
2,4,original_post,bd5j7h,My boyfriend has a friend that he's been very ...,My boyfriend has a friend that he's been very ...,2
3,6,original_post,d7m3uc,I need advice about getting angry at my brothe...,I need advice about getting angry at my brothe...,3
4,8,original_post,ck31nx,I need perspective on a conversation that left...,I need perspective on a conversation that left...,4


## Config
`stopword_files` is optional in this preprocessor - drop the arg entirely if you don't have custom stopword files; spaCy's own stopword list is already applied by default.

In [12]:
MODELS_DIR = "models_prompt"
NUM_ITERS = 1500
TOPN_WORDS = 15
MIN_DOC_WORDS = 2
TOP_N_REPRESENTATIVE = 10

LLM_PROVIDER = "openai"
LLM_MODEL = "gpt-5"
LLM_API_KEY = os.getenv("OPENAI_API_KEY")     # plain string, not a client
LLM_SERVER = "https://eu.api.openai.com/v1"   # your custom base_url

os.makedirs(MODELS_DIR, exist_ok=True)

In [13]:
import os
os.environ["OPENAI_API_KEY"] = "

In [14]:
def build_preprocessor():
    return SimpleTMPreprocessor(
        spacy_model="en_core_web_lg",
        spacy_disable=["parser", "ner"],
        valid_pos=["NOUN", "VERB", "ADJ"],
        min_df=3,
        max_df=0.6,
        logger=logger,
        # stopword_files=["path/to/your_stops.txt"],  # optional
    )

## Build reference corpus (lemmatized, for coherence)

In [15]:
df_lemmas = build_preprocessor().fit_transform(
    df, text_col="text", id_col="id", compute_bow=False, compute_tfidf=False,
)
reference_corpus = df_lemmas["lemmas"].tolist()

2026-09-01 15:51:36,659  INFO  Loading spaCy model 'en_core_web_lg'


## Optimize number of topics
Trains one model per K in `TOPIC_RANGE`, scores each by coherence (c_NPMI), smooths the coherence-vs-K curve, and returns the peak(s). Every model is saved under `OPTIMIZE_DIR/model_{k}_topics/` regardless of whether it's selected, so nothing is wasted - and a `coherence_curve.html` plot is written to `OPTIMIZE_DIR` for a visual check.

This trains `len(TOPIC_RANGE)` full models, so it costs proportionally more time than a single fixed-K run - narrow the range once you have a rough sense of where coherence peaks.

In [16]:
TOPIC_RANGE = range(5, 31, 5)  # adjust to your expected granularity
OPTIMIZE_DIR = Path(MODELS_DIR) / "optimize"

opt_results = LDATopicModel.optimize_num_topics(
    data=df[["id", "text"]].to_dict(orient="records"),
    base_path=OPTIMIZE_DIR,
    topic_range=TOPIC_RANGE,
    smoothing_window=3,
    n_best=3,
    logger=logger,
    num_iters=NUM_ITERS,
    topn=TOPN_WORDS,
    min_doc_words=MIN_DOC_WORDS,
    preprocessor=build_preprocessor(),
    reference_corpus=reference_corpus,
)

print("k -> mean coherence:")
for k, c in opt_results["scores"]:
    print(f"  {k:>3}: {c:.4f}")
print("\nSelected (coherence peaks):", opt_results["selected"])

2026-09-01 15:51:56,563  INFO  Loading spaCy model 'en_core_web_lg'
2026-09-01 15:51:57,846  INFO  [optimize] Training model with k=5 topics -> models_prompt/optimize/model_5_topics
2026-09-01 15:51:57,848  INFO  Old model moved to models_prompt/optimize/model_5_topics_old
2026-09-01 15:51:57,849  INFO  Preprocessing training data.
2026-09-01 15:52:11,893  INFO  Trainable: 1000  Short (fallback): 0
2026-09-01 15:52:11,894  INFO  Training LDA...
/home/muti/miniconda3/envs/testenv/lib/python3.11/site-packages/tomotopy/models.py:637: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  return self._train(iterations, workers, parallel, freeze_topics, callback_interval, callback)
2026-09-01 15:52:11,993  INFO    iter    0/1500  ll=-7.6128  perp=2024.0
2026-09-01 15:52:12,094  INFO    iter  100/1500  ll=-7.1024  perp=1214.9.12it/s]
2026-09-01 15:52:12,197  INFO    iter  200/1500  ll=-7.0635  perp=1168.5.81it/s]
2026-09-01 15:52:12,301  INFO    iter  300/15

k -> mean coherence:
    5: -0.0088
   10: 0.0118
   15: 0.0037
   20: 0.0016
   25: -0.0212
   30: -0.0272

Selected (coherence peaks): [{'k': 15, 'mean_coherence': 0.003741284146339376, 'model_path': 'models_prompt/optimize/model_15_topics'}]


## Pick K
Defaults to the top-coherence peak from `opt_results["selected"]`. Open `OPTIMIZE_DIR/coherence_curve.html` and override `CHOSEN_K` by hand if a different peak makes more sense qualitatively (e.g. topic counts that split cleanly vs. ones that don't).

In [17]:
best = max(opt_results["selected"], key=lambda s: s["mean_coherence"])
CHOSEN_K = best["k"]
#CHOSEN_K = 15
model_path = Path(best["model_path"])
print(f"Using K={CHOSEN_K}  (mean coherence={best['mean_coherence']:.4f})  ->  {model_path}")

Using K=15  (mean coherence=0.0037)  ->  models_prompt/optimize/model_15_topics


## Load the chosen model
`optimize_num_topics` already trained and saved this model - no need to retrain.

In [18]:
df_corpus = df[["id", "text"]]
lda = LDATopicModel.load(model_path, corpus=df_corpus)

coh = np.load(model_path / "TMmodel" / "topic_coherence.npy")
logger.info(f"  mean coherence (c_npmi) = {float(np.mean(coh)):.4f}")

2026-09-01 15:53:56,639  INFO  Loading Tomotopy model from models_prompt/optimize/model_15_topics/model.bin
2026-09-01 15:53:56,669  INFO  Thetas restored from disk: (1000, 15)
2026-09-01 15:53:56,672  INFO  Topic keys restored: 15 topics
2026-09-01 15:53:56,672  INFO  Model loaded successfully.
2026-09-01 15:53:56,673  INFO    mean coherence (c_npmi) = 0.0037


## Generate topic labels via LLM
**Important:** set `llm_provider` / `llm_model_type` / `llm_api_key` on `lda.tm`, not on `lda`. `train()` builds `lda.tm` internally and it snapshots these values at that point - setting them on `lda` afterward has no effect.

In [19]:
labels_file = model_path / "TMmodel" / "tpc_labels.txt"

def _is_placeholder(labels):
    return labels == [f"Topic {i}" for i in range(len(labels))]

existing = labels_file.read_text(encoding="utf-8").splitlines() if labels_file.exists() else []

if existing and not _is_placeholder(existing):
    labels = existing
    logger.info(f"Found existing tpc_labels.txt ({len(labels)} labels) - skipping labeling.")
else:
    logger.info(f"[{LLM_PROVIDER}] Generating topic labels via {LLM_MODEL} ...")
    lda.tm.llm_provider = LLM_PROVIDER
    lda.tm.llm_model_type = LLM_MODEL
    lda.tm.llm_api_key = LLM_API_KEY
    lda.tm.llm_server = LLM_SERVER
    results = lda.tm.generate_topic_outputs(task="label", topn=TOPN_WORDS)
    labels = [lbl for _, lbl in sorted(results)]
    labels_file.write_text("\n".join(labels), encoding="utf-8")

for i, lbl in enumerate(labels):
    print(f"T{i}: {lbl}")

2026-09-01 15:54:00,921  INFO  [openai] Generating topic labels via gpt-5 ...
2026-09-01 15:54:00,922  INFO  -- -- -- Topic model object (TMmodel) successfully created
2026-09-01 15:54:00,998  INFO  Effective LLM provider used: openai, model: gpt-5, server: https://eu.api.openai.com/v1
2026-09-01 15:54:00,999  INFO  Using provider: openai
2026-09-01 15:54:02,748  INFO  HTTP Request: GET https://eu.api.openai.com/v1/models "HTTP/1.1 200 OK"
2026-09-01 15:54:02,789  INFO  Using OpenAI API with model: gpt-5, base_url: https://eu.api.openai.com/v1


Cache key: b696974c35b4fd36575dba311b67ba44
Cache key: 6b1fcab7ec5c23851b85141ec74d450a
Cache key: 3b09b2d155f3d34c8b7af2f007b47ccd
Cache key: d89ab7c6ab2662565911c3b972ce625d
Cache key: 5bd36bacc65a4794dbe0cf481b206e2e
Cache key: 01dcff579269384ea37fedbb151426ea
Cache key: 9aac863c871210b37317e9771133788b
Cache miss: computing results...
Cache key: e45b32278ca81177abcf483cf6917c6c
Cache miss: computing results...
Cache key: b40db0dc357354834ae964e96f88709d
Cache miss: computing results...
Cache miss: computing results...
Cache miss: computing results...
Cache miss: computing results...
Cache miss: computing results...
Cache miss: computing results...
Cache key: 2fab761a0b3fefc7a6be85321573eb4c
Cache key: 7542836c90f7913e900bb79f09807c6d
Cache miss: computing results...
Cache key: 1a9749f209e1a244e8168a99952b7218
Cache key: 701a56af749a124c928e33f16f7c2d0e
Cache key: 0a96a58507e2a0f1a6e6bb3fb73dba17
Cache key: 862bdfc6238b317944a55a219bddc906
Cache miss: computing results...
Cache miss

2026-09-01 15:54:03,424  INFO  HTTP Request: POST https://eu.api.openai.com/v1/chat/completions "HTTP/1.1 403 Forbidden"
2026-09-01 15:54:03,425  INFO  HTTP Request: POST https://eu.api.openai.com/v1/chat/completions "HTTP/1.1 403 Forbidden"
2026-09-01 15:54:03,430  INFO  HTTP Request: POST https://eu.api.openai.com/v1/chat/completions "HTTP/1.1 403 Forbidden"
2026-09-01 15:54:03,431  INFO  HTTP Request: POST https://eu.api.openai.com/v1/chat/completions "HTTP/1.1 403 Forbidden"
2026-09-01 15:54:03,432  INFO  HTTP Request: POST https://eu.api.openai.com/v1/chat/completions "HTTP/1.1 403 Forbidden"
2026-09-01 15:54:03,435  INFO  HTTP Request: POST https://eu.api.openai.com/v1/chat/completions "HTTP/1.1 403 Forbidden"
2026-09-01 15:54:03,439  INFO  HTTP Request: POST https://eu.api.openai.com/v1/chat/completions "HTTP/1.1 403 Forbidden"
2026-09-01 15:54:03,439  INFO  HTTP Request: POST https://eu.api.openai.com/v1/chat/completions "HTTP/1.1 403 Forbidden"
2026-09-01 15:54:03,450  INFO  H

T0: Relationships
T1: etiquette
T2: Relationships
T3: boundaries
T4: Adoption
T5: Money
T6: Roommates
T7: Etiquette
T8: Workplace
T9: Photos
T10: household
T11: Education
T12: Weddings
T13: Cooking
T14: Boundaries


## Export document-topic assignments + representative examples + keywords

In [20]:
thetas = lda.get_thetas()
df_out = df.copy()
df_out["dominant_topic"] = np.argmax(thetas, axis=1)
df_out["topic_label"] = df_out["dominant_topic"].map(lambda t: labels[t] if t < len(labels) else "")
df_out["topic_prob"] = thetas.max(axis=1)
df_out["topic_composition"] = [
    {i: round(float(p), 4) for i, p in enumerate(row)} for row in thetas
]

Kt = thetas.shape[1]
uniform_threshold = (1.0 / Kt) + 1e-4
df_out["low_confidence"] = df_out["topic_prob"] <= uniform_threshold
logger.info(
    f"  {int(df_out['low_confidence'].sum()):,} / {len(df_out):,} docs flagged "
    f"low_confidence (max theta <= {uniform_threshold:.4f})."
)

assignments_path = model_path / "document_topic_assignments.csv"
df_out[["id", "prompt_text", "dominant_topic", "topic_label",
        "topic_prob", "low_confidence", "topic_composition"]].to_csv(assignments_path, index=False)
print("saved:", assignments_path)

2026-09-01 15:55:24,295  INFO    0 / 1,000 docs flagged low_confidence (max theta <= 0.0668).


saved: models_prompt/optimize/model_15_topics/document_topic_assignments.csv


In [21]:
# representative persona_text examples per topic, ranked by theta
lda.tm.get_most_representative_per_tpc(
    sparse.csr_matrix(thetas), topn=TOP_N_REPRESENTATIVE, get_text=False,
)
text_by_id = dict(zip(df_out["id"], df_out["prompt_text"]))

records = []
for t, docs in enumerate(lda.tm._most_representative_docs):
    for rank, (doc_id, _text, theta) in enumerate(docs, start=1):
        records.append({
            "topic_id": t,
            "topic_label": labels[t] if t < len(labels) else "",
            "rank": rank,
            "theta": round(float(theta), 4),
            "id": doc_id,
            "persona_text": text_by_id.get(doc_id, ""),
        })
repr_path = model_path / "topic_representative_examples.csv"
pd.DataFrame(records).to_csv(repr_path, index=False)
print("saved:", repr_path)

saved: models_prompt/optimize/model_15_topics/topic_representative_examples.csv


In [22]:
topic_words = lda.get_topic_keys()
keywords_path = model_path / "topic_keywords.csv"
pd.DataFrame({
    "topic_id": range(len(labels)),
    "topic_label": labels,
    "n_docs": df_out["dominant_topic"].value_counts().reindex(range(len(labels)), fill_value=0).values,
    "top_words": [", ".join(w) for w in topic_words],
}).to_csv(keywords_path, index=False)
print("saved:", keywords_path)

saved: models_prompt/optimize/model_15_topics/topic_keywords.csv
